<a href="https://colab.research.google.com/github/hitarthi45/Artificial-Intelligence/blob/main/(Exp_12)Collaborative_based_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
ratings = pd.read_csv("ratings.csv")
# ================= USER-ITEM MATRIX =================
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)
# ================= USER SIMILARITY =================
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
# ================= RECOMMEND FUNCTION =================
def recommend_movies(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found!"

    # Similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Weighted ratings
    weighted_ratings = np.zeros(user_item_matrix.shape[1])

    for sim_user, score in similar_users.items():
        weighted_ratings += score * user_item_matrix.loc[sim_user].values

    # Normalize
    weighted_ratings /= similar_users.sum()

    # Get unseen movies
    user_rated = user_item_matrix.loc[user_id]
    unseen_movies = user_rated[user_rated == 0].index

    scores = list(zip(unseen_movies, weighted_ratings[user_item_matrix.columns.get_indexer(unseen_movies)]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommended_ids = [movie for movie, _ in scores[:top_n]]

    return recommended_ids
print(recommend_movies(user_id=1))

[1200, 1610, 541, 589, 1036]


###POST LAB

In [2]:
!wget -nc http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -n ml-100k.zip

--2026-04-28 03:43:52--  http://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-100k.zip [following]
--2026-04-28 03:43:52--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  12.1MB/s    in 0.4s    

2026-04-28 03:43:53 (12.1 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml

In [5]:
# Load movie titles from u.item
movie_cols = ['movieId', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure',
              'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
              'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies_df = pd.read_csv('ml-100k/u.item', sep='|', names=movie_cols, encoding='latin-1')

# Select only movieId and title for recommendation display
movies_df = movies_df[['movieId', 'title']]
display(movies_df.head())

,movieId,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [9]:
import numpy as np
import pandas as pd
# The following variables are assumed to be globally available from cell `7QckjZXcBoGr`:
# `user_item_matrix`, `user_similarity_df`

def recommend_movies(user_id, movies_df, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found!"

    # Similar users (excluding the user itself)
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Handle cases where no similar users or all scores are zero
    sum_scores = similar_users.sum()
    if sum_scores == 0:
        return [] # No similar users with positive scores to base recommendations on

    # Weighted ratings
    weighted_ratings = np.zeros(user_item_matrix.shape[1])
    for sim_user, score in similar_users.items():
        weighted_ratings += score * user_item_matrix.loc[sim_user].values
    weighted_ratings /= sum_scores

    # Get unseen movies for the target user
    user_rated = user_item_matrix.loc[user_id]
    unseen_movies = user_rated[user_rated == 0].index

    # Filter unseen movies that are actually present in the user-item matrix columns
    unseen_movies_in_matrix = [m for m in unseen_movies if m in user_item_matrix.columns]

    # Get recommendation scores for these unseen movies
    # Use .get_indexer to safely map movie IDs to column indices
    scores_for_unseen = weighted_ratings[user_item_matrix.columns.get_indexer(unseen_movies_in_matrix)]

    # Combine movie IDs with their scores and sort
    scores = list(zip(unseen_movies_in_matrix, scores_for_unseen))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Get the top N recommended movie IDs
    recommended_ids = [movie_id for movie_id, _ in scores[:top_n]]

    # Map movie IDs to titles using the movies_df
    if not movies_df.empty and 'movieId' in movies_df.columns and 'title' in movies_df.columns:
        recommended_titles = movies_df[movies_df['movieId'].isin(recommended_ids)]['title'].tolist()
    else:
        recommended_titles = [f"Movie ID {mid}" for mid in recommended_ids] # Fallback

    return recommended_titles

In [10]:
# Recommend top 10 movies for User 1 using the updated function
recommended_titles_user_1 = recommend_movies(user_id=1, movies_df=movies_df, top_n=10)

In [11]:
# Convert the list of recommended titles into a DataFrame for tabular display
recommended_titles_df = pd.DataFrame(recommended_titles_user_1, columns=['Recommended Movie Title'])
display(recommended_titles_df)

,Recommended Movie Title
0,Mortal Kombat (1995)
1,"Wild Bunch, The (1969)"
2,Amityville: Dollhouse (1996)
3,White Squall (1996)
4,Drop Dead Fred (1991)
5,Kim (1950)
6,When a Man Loves a Woman (1994)
7,Falling in Love Again (1980)
8,"Truth or Consequences, N.M. (1997)"
